[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_01_setup/01_environment_check.ipynb)

# Week 1 · Environment check

Working in Colab? Follow the [Colab setup instructions](https://github.com/fiit-ba/zneus-2026/blob/main/labs/week_01_setup/README.md#google-colab) to open your private repository, then run the setup cell below.

Before we train anything, let us make sure your tools work: the Python environment, PyTorch, the accelerator (if you have one), Weights & Biases (W&B) and Kaggle. Run this notebook **once on your own machine** and **once in Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU). Every cell prints `OK` or `WARN` lines, so you can see at a glance what needs attention.

**Week 1 hand-in:** one successfully scored Kaggle warm-up upload under your AIS login (section 6). No notebook, commit link or W&B link submission is required. The other steps help you practise the tools.

## What you will do
1. **Versions**: check Python and the packages we use during the semester.
2. **Device detection**: find out whether PyTorch sees a CUDA GPU, an Apple GPU (MPS) or only the CPU.
3. **Micro-benchmark**: time a large matrix multiplication on the CPU and on your accelerator.
4. **Autograd smoke test**: let PyTorch compute a derivative for you.
5. **W&B practice run**: log a fake training curve locally, or optionally try online logging to see it in W&B.
6. **Kaggle warm-up submission**: build a random `submission.csv` and upload it to the warm-up competition. This is the only `TODO` in this notebook.
7. **Summary**: a checklist of what passed.

## How to work through this notebook
- Run the cells **in order** (Shift+Enter). Later cells reuse variables from earlier ones.
- The one place that needs your input is marked with a `# TODO` comment and `...`. **Replace every `...` with your own code.**
- The verification cell after the TODO tells you whether your submission file has the right shape.
- If a cell prints `WARN`, read the message: it tells you what to fix (usually `uv sync`, `wandb login` or a Colab runtime setting).

In [1]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [2]:
import importlib.metadata
import math
import os
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# We tick these off as we go; the summary at the end prints the checklist.
checks = {
    "versions": False,
    "device detected": False,
    "matmul benchmark": False,
    "autograd": False,
    "W&B test run": False,
    "Kaggle CSV validated locally": False,
}

In [3]:
# --- Configuration ---
SEED = 42
DATA_DIR = Path(os.environ.get("ZNEUS_DATA_DIR", "data"))

torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

device: cpu


## 1. Versions

The course environment is defined in `pyproject.toml` and installed with `uv sync` (see [SETUP.md](SETUP.md)). We use Python 3.12, PyTorch 2.x and a handful of standard libraries. `importlib.metadata.version` reads the installed version of a package without importing it.

In [4]:
print(f"{'python':<14}{platform.python_version()}   ({sys.executable})")
print(f"{'platform':<14}{platform.platform()}")

packages = ["torch", "torchvision", "numpy", "pandas", "matplotlib", "scikit-learn", "wandb"]
missing = []
for name in packages:
    try:
        print(f"{name:<14}{importlib.metadata.version(name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{name:<14}MISSING")
        missing.append(name)

python_ok = sys.version_info[:2] in {(3, 12), (3, 13)}
torch_ok = torch.__version__.startswith("2.")
if not python_ok:
    print(f"WARN  Python {platform.python_version()} found; the course is tested on 3.12. Use `uv sync` and run Jupyter with `uv run jupyter lab`.")
if not torch_ok:
    print(f"WARN  torch {torch.__version__} found; expected 2.x.")
if missing:
    print(f"WARN  missing packages: {missing}. Run `uv sync` in the repository root (Colab: `pip install {' '.join(missing)}`).")
if python_ok and torch_ok and not missing:
    print("OK    all packages found")
    checks["versions"] = True

python        3.12.14   (c:\Other\Fiit-5-sem\ZNEUS\zneus-2026\.venv\Scripts\python.exe)
platform      Windows-11-10.0.26200-SP0
torch         2.14.0+cpu
torchvision   0.29.0+cpu
numpy         2.5.3
pandas        3.0.5
matplotlib    3.11.1
scikit-learn  1.9.0
wandb         0.30.0
OK    all packages found


## 2. Device detection

PyTorch can run the same code on different hardware. A **device** string tells it where a tensor lives:

- `"cuda"`: an NVIDIA GPU (your own, or the free T4 in Colab),
- `"mps"`: the GPU of an Apple Silicon Mac (Metal Performance Shaders),
- `"cpu"`: always available and perfectly fine for weeks 2 to 5.

From week 6 on, training on a GPU is much more comfortable; if you do not have one, use Colab. The cell below prints what PyTorch found. On a machine with an NVIDIA driver it also runs `nvidia-smi`, the command-line tool that shows GPU utilisation and memory.

In [5]:
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_memory_gb = props.total_memory / 2**30
    print(f"OK    CUDA GPU: {props.name}, {gpu_memory_gb:.1f} GB memory, CUDA {torch.version.cuda}, {torch.cuda.device_count()} device(s)")
elif device == "mps":
    mps_memory_gb = torch.mps.recommended_max_memory() / 2**30
    print(f"OK    Apple GPU via MPS ({platform.machine()}), recommended max memory {mps_memory_gb:.1f} GB")
else:
    print(f"WARN  no GPU found, PyTorch will use the CPU ({os.cpu_count()} cores, {torch.get_num_threads()} threads).")
    print("      Fine for weeks 2-5. For the later weeks open this notebook in Colab and pick a GPU runtime.")
    if IN_COLAB:
        print("      In Colab: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU, then Runtime -> Restart session.")

if shutil.which("nvidia-smi"):
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                             capture_output=True, text=True, timeout=20, check=False)
        smi_report = (out.stdout or out.stderr).strip()
        print("nvidia-smi:", smi_report)
    except (OSError, subprocess.SubprocessError) as e:
        print("WARN  nvidia-smi could not be run:", e)

checks["device detected"] = True

WARN  no GPU found, PyTorch will use the CPU (8 cores, 8 threads).
      Fine for weeks 2-5. For the later weeks open this notebook in Colab and pick a GPU runtime.


## 3. Micro-benchmark: how much faster is your accelerator?

Almost all the work in a neural network is matrix multiplication. Multiplying two $n \times n$ matrices costs about $2n^3$ floating-point operations, so a single $2048 \times 2048$ product is roughly 17 GFLOP. Doing it ten times on the CPU and ten times on the accelerator gives a rough but honest idea of the speed-up you can expect.

Two details make GPU timing tricky:

- **Asynchronous execution.** GPU operations are *queued*; Python moves on before the result exists. To measure the real time we must wait for the queue to drain with `torch.cuda.synchronize()` (or `torch.mps.synchronize()`).
- **Warm-up.** The first call pays for memory allocation and kernel loading, so we do one untimed multiplication first.

In [6]:
N = 2048
REPS = 10


def synchronize(dev: str) -> None:
    """Wait until every queued operation on `dev` has finished."""
    if dev == "cuda":
        torch.cuda.synchronize()
    elif dev == "mps":
        torch.mps.synchronize()


def matmul_seconds(dev: str, n: int = N, reps: int = REPS) -> float:
    """Average wall-clock seconds of one (n x n) @ (n x n) float32 matmul on `dev`."""
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    _ = a @ b                      # warm-up, not timed
    synchronize(dev)
    t0 = time.perf_counter()
    for _ in range(reps):
        c = a @ b
    synchronize(dev)
    elapsed = time.perf_counter() - t0
    return elapsed / reps


gflop = 2 * N**3 / 1e9
t_cpu = matmul_seconds("cpu")
gflops_cpu = gflop / t_cpu
print(f"{'cpu':<6}{t_cpu * 1e3:8.1f} ms per {N}x{N} matmul   ({gflops_cpu:7.0f} GFLOP/s)")

if device != "cpu":
    t_dev = matmul_seconds(device)
    gflops_dev = gflop / t_dev
    speedup = t_cpu / t_dev
    print(f"{device:<6}{t_dev * 1e3:8.1f} ms per {N}x{N} matmul   ({gflops_dev:7.0f} GFLOP/s)")
    print(f"OK    {device} is {speedup:.1f}x faster than the CPU on this operation")
else:
    print("OK    CPU benchmark done (no accelerator to compare with)")
checks["matmul benchmark"] = True

cpu      175.7 ms per 2048x2048 matmul   (     98 GFLOP/s)
OK    CPU benchmark done (no accelerator to compare with)


**3.1 Check your understanding.** Why would the timing be wrong (and suspiciously good) if we removed the `synchronize(dev)` call *before* reading `time.perf_counter()` the second time?

_Your answer here._

## 4. Autograd smoke test

PyTorch records every operation performed on a tensor with `requires_grad=True` and can then compute derivatives automatically with `.backward()`. This is the engine behind training neural networks; we will implement it by hand in week 3, and use it from week 4 on.

For $y = x^2 + 2x$ we have $\frac{dy}{dx} = 2x + 2$, so at $x = 3$ the derivative must be exactly $8$.

In [7]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2 * x
y.backward()                       # fills x.grad with dy/dx

print(f"y = {y.item():.1f},  dy/dx at x=3: {x.grad.item():.1f}")
assert x.grad.item() == 8.0, f"expected 8.0, got {x.grad.item()}"
print("OK    autograd works")
checks["autograd"] = True

y = 15.0,  dy/dx at x=3: 8.0
OK    autograd works


## 5. Weights & Biases practice run

[Weights & Biases](https://wandb.ai) (W&B) is an experiment tracker: your training script logs metrics (loss, accuracy, learning rate, ...) and W&B draws the curves and stores the configuration of every run. **A public W&B project link is required only for the semester project.**

The practice cell below defaults to **offline mode**, so you can run it without an account. It saves a fake training curve in a local `wandb/` folder. No W&B link or upload is required for this lab.

**Optional: try online logging**

1. Create an account at [wandb.ai](https://wandb.ai/site) with your `@stuba.sk` e-mail.
2. Log in from the terminal, once per machine: `uv run wandb login` and paste the API key from [wandb.ai/authorize](https://wandb.ai/authorize). In Colab, the cell below asks for the key interactively.
3. Set `wandb_mode = "online"` in the cell below and run it. It creates the project `zneus-2026` on your account and logs 20 steps of a fake, decreasing loss.
4. Open the printed run URL and inspect the curves. Your lab practice runs can stay private.

**For the semester project:** make the W&B project public and include its URL with your project submission. If you logged offline, upload the relevant runs first with `uv run wandb sync wandb/offline-run-*`.

Suggested naming: W&B project `zneus-2026`, one run per experiment, and a meaningful run name (for example `week05-mlp-lr1e-3`). Never hard-code your user name (the *entity*) or API key in the code.


In [11]:
import wandb

wandb_mode = os.environ.get("WANDB_MODE", "online")  # set to "online" to try the web dashboard
if wandb_mode == "online":
    try:
        logged_in = wandb.login(prompt=False)      # True if a valid API key is stored, False if none is found
    except Exception as e:                          # e.g. a stored key that the server rejects
        logged_in = False
        print("WARN  W&B login failed:", e)
    if not logged_in:
        print("WARN  no valid W&B API key found. Run `uv run wandb login` in a terminal (or paste the key when asked below),")
        print("      or set WANDB_MODE=offline to skip logging in.")

run = wandb.init(
    project="zneus-2026",
    name="week01-hello",
    mode=wandb_mode,
    config={"device": device, "torch": torch.__version__, "python": platform.python_version(), "colab": IN_COLAB},
)

g = torch.Generator().manual_seed(SEED)
for step in range(20):
    decay = 2.0 * math.exp(-0.25 * step)
    noise = 0.05 * torch.rand((), generator=g).item()
    fake_loss = decay + noise
    fake_accuracy = 1.0 - fake_loss / 2.5
    wandb.log({"train/loss": fake_loss, "val/accuracy": fake_accuracy}, step=step)

if run.url:
    print("OK    practice run logged. Open the curves here:", run.url)
else:
    print(f"OK    run logged in {wandb_mode} mode (no URL). Local files: {run.dir}")
run.finish()
checks["W&B test run"] = True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ivank\_netrc.
wandb: Currently logged in as: xkuchinka (xkuchinka-xkuchinka) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


OK    practice run logged. Open the curves here: https://wandb.ai/xkuchinka-xkuchinka/zneus-2026/runs/690a0pl1


train/loss,█▆▅▄▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/accuracy,▁▃▄▅▅▆▇▇▇▇▇█████████
train/loss,0.04868
val/accuracy,0.98053


## 6. Kaggle warm-up submission

This week's goal is to upload one valid CSV and find its score in [the week 1 competition](https://www.kaggle.com/competitions/zneus-2026-week-1-kaggle-warm-up). **No model training is required.** A successfully scored submission completes this warm-up; there is no minimum score or leaderboard rank to reach.

**One-time setup**

1. Create a Kaggle account and complete any verification Kaggle requests.
2. [Join the week 1 warm-up with this invitation link](https://www.kaggle.com/t/5e8b5441c9144adbaa973dceac384939) and read and accept the competition rules. Joining is restricted to invitation-link holders; keep the link within the course.
3. Set your **competition team name** to your **AIS login**, so the instructor can identify your submission. Participate individually.
4. Optional: download `sample_submission.csv` from the competition's [Data tab](https://www.kaggle.com/competitions/zneus-2026-week-1-kaggle-warm-up/data) into `DATA_DIR / "kaggle"`. The cell below uses its `id` column. Without the file it uses the same IDs, `0..999`.

**Deadline:** 30 September 2026 at 23:59 Europe/Bratislava. You may upload up to five submissions per day through the website; the Kaggle command-line tool is optional.

For a quick browser-only upload test, download `sample_submission.csv`, save a copy as `submission.csv`, and follow the upload steps in section 6.3. The notebook exercise below teaches you to generate the same file yourself.

**6.1 Build a random submission.** Fill in the `TODO` below:

- Create a random-number generator with a fixed seed: `g = torch.Generator().manual_seed(SEED)`.
- Draw one standard-normal number per ID with `torch.randn(len(ids), generator=g)`.
- Build a `pandas.DataFrame` with the two columns `id` and `prediction` (convert the tensor with `.numpy()`).
- The cell writes `submission.csv` in the notebook's current working directory. Keep `index=False` to avoid an extra index column.

The CSV must contain exactly **1,000 data rows**, with columns `id,prediction`, every ID from `0` to `999` once, and a finite numeric prediction for each.


In [14]:
KAGGLE_JOIN_URL = "https://www.kaggle.com/t/5e8b5441c9144adbaa973dceac384939"
KAGGLE_URL = "https://www.kaggle.com/competitions/zneus-2026-week-1-kaggle-warm-up"

sample_path = Path("C:/Other/Fiit-5-sem/ZNEUS/sample_submission.csv")
if sample_path.exists():
    ids = pd.read_csv(sample_path)["id"].to_numpy()
    print(f"using the {len(ids)} ids from {sample_path}")
else:
    ids = np.arange(1000)
    print(f"{sample_path} not found: using ids 0..{len(ids) - 1}")

# TODO: seeded generator -> len(ids) random predictions -> DataFrame with columns id, prediction
g = g = torch.Generator().manual_seed(SEED)
preds = torch.randn(len(ids), generator=g)
submission = pd.DataFrame({"id": ids, "prediction": preds.numpy()})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("shape:", submission.shape)

using the 1000 ids from C:\Other\Fiit-5-sem\ZNEUS\sample_submission.csv
   id  prediction
0   0    1.926915
1   1    1.487284
2   2    0.900717
3   3   -2.105521
4   4    0.678418
shape: (1000, 2)


**6.2 Verify the file.** The next cell re-reads `submission.csv` from disk and checks the format Kaggle expects. A local check does not upload the file; complete section 6.3 afterwards.


In [15]:
reloaded = pd.read_csv("submission.csv")
assert list(reloaded.columns) == ["id", "prediction"], f"columns must be ['id', 'prediction'], got {list(reloaded.columns)}"
assert reloaded.shape == (1000, 2), f"expected shape (1000, 2), got {reloaded.shape}"
assert reloaded["id"].is_unique, "ids must be unique"
assert np.array_equal(np.sort(reloaded["id"].to_numpy()), np.arange(1000)), "ids must cover 0..999 exactly"
assert np.isfinite(reloaded["prediction"]).all(), "predictions must be finite numbers"
print(f"OK    submission.csv: shape {reloaded.shape}, columns {list(reloaded.columns)}")
print(f"File: {Path('submission.csv').resolve()}")
print(f"Next: open {KAGGLE_URL} -> Submit Prediction and upload this file.")
checks["Kaggle CSV validated locally"] = True


OK    submission.csv: shape (1000, 2), columns ['id', 'prediction']
File: C:\Other\Fiit-5-sem\ZNEUS\zneus-2026\labs\week_01_setup\submission.csv
Next: open https://www.kaggle.com/competitions/zneus-2026-week-1-kaggle-warm-up -> Submit Prediction and upload this file.


**6.3 Upload and check the score.**

1. Find the `submission.csv` path printed above. In Colab, open the **Files** sidebar, refresh it if needed, then use the file's menu to **Download** it to your computer.
2. Open [the competition](https://www.kaggle.com/competitions/zneus-2026-week-1-kaggle-warm-up) and click **Submit Prediction**. If you cannot join, use [the invitation link](https://www.kaggle.com/t/5e8b5441c9144adbaa973dceac384939) first.
3. Select `submission.csv`, enter the description `Week 1 upload test`, and submit. From week 5 on, this description box is how your commit link reaches the instructor; week 1 needs no link, so any text will do.
4. Wait for processing to finish. Open **Submissions** and confirm it says **Complete**, has a numeric public score, and shows no error. Check that your competition team name is your AIS login.

The supplied sample file should score approximately **1.13930** on the public leaderboard. Random predictions generated by your environment may differ; a particular score is not required. If Kaggle rejects the CSV, check that you selected the file verified above and that it contains exactly `id,prediction` with all 1,000 IDs.

**Optional improvement:** the synthetic target is `prediction = id / 999`. Replace the random predictions with that formula and submit again; exact predictions have RMSE 0. Lower RMSE is better. This extra upload is optional.


## 7. Summary

The checklist below checks your setup and the local CSV file. **Week 1 is complete when Kaggle successfully scores your upload under your AIS login.** The notebook cannot check that upload for you; open Kaggle's **Submissions** page to confirm it.

Keep your notebook for practice and run it in Colab with a GPU runtime to compare the benchmark numbers. No notebook, commit link or W&B link submission is required.


In [ ]:
print(f"Week 1 setup checklist  (device: {device}, python {platform.python_version()}, torch {torch.__version__})")
for name, ok in checks.items():
    print(f"  [{'x' if ok else ' '}] {name}")

if all(checks.values()):
    print("\nOK    setup checks passed.")
else:
    print("\nWARN  some setup checks did not pass; scroll up to the first WARN, fix it, then re-run the notebook.")
print("\nWeek 1 hand-in:")
print(f"  Upload submission.csv at {KAGGLE_URL} and check Submissions for a numeric score with no error.")
print("  Use your AIS login as the competition team name. No minimum score or rank is required.")
print("  Deadline: 30 September 2026, 23:59 Europe/Bratislava.")
print("  No notebook, commit link or W&B link submission is required.")
print("\nSetup practice:")
print("  Save your notebook to your private repository so you can return to it later.")
print("  Run this notebook in Colab with a GPU runtime and compare the benchmark numbers.")
